# Real-Time Stock Market Stream Analysis

This notebook explores stock-price streams with Apache Kafka and Apache Spark Structured Streaming. It includes windowed price analysis, identification of price movements, visual summaries, and an optional InfluxDB output.


In [ ]:
from pyspark.sql.functions import explode
from pyspark.sql.functions import split

from pyspark.context import SparkContext
from pyspark.conf import SparkConf
from pyspark.sql.session import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("MyApp") \
    .config("spark.jars.packages", 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.0.0') \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .getOrCreate()


Be sure to start the stream on Kafka!

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, TimestampType, DateType

schema = StructType(
      [
        StructField("name", StringType(), False),
        StructField("price", DoubleType(), False),
        StructField("timestamp", TimestampType(), False),
      ]
    )

In [ ]:
import os
kafka_server = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
from pyspark.sql.functions import from_json

lines = (spark.readStream                        # Get the DataStreamReader
  .format("kafka")                                 # Specify the source format as "kafka"
  .option("kafka.bootstrap.servers", kafka_server) # Configure the Kafka server name and port
  .option("subscribe", "stock")                       # Subscribe to the "en" Kafka topic 
  .option("startingOffsets", "earliest")           # The start point when a query is started
  .option("maxOffsetsPerTrigger", 100)             # Rate limit on max offsets per trigger interval
  .load()
  .select(from_json(col("value").cast("string"), schema).alias("parsed_value"))
# Load the DataFrame
)
df = lines.select("parsed_value.*")


## Streaming analysis tasks



### Task 1: Rank stocks by average price in a time window


In [ ]:
# Install dependencies from requirements.txt before running this notebook.


In [ ]:
import time
from pyspark.sql.streaming import StreamingQueryException
# Visualization code
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, col
from pyspark.sql import functions as F

In [ ]:
# Group the data by a 20-minute window and stock name, and calculate the average price
windowedDF = df.groupBy(
    window(col("timestamp"), "5 minutes"),
    "name"
).avg("price").withColumnRenamed("avg(price)", "avg_price")

# Now, since you want the 5 most valuable stocks, we will have to do a ranking
# For streaming DataFrame, we would need to write the results in update mode to a query sink that supports updates (like a memory sink or a Delta table)

# We need to define an aggregation query
topStocksQuery = windowedDF \
    .writeStream \
    .outputMode("complete") \
    .format("console") \
    .foreachBatch(lambda df, epoch_id: df.orderBy(F.desc("avg_price")).limit(10).show(truncate=False)) \
    .start()

topStocksQuery.awaitTermination(timeout=5)
topStocksQuery.stop()


In [ ]:
# Step 2: Define the streaming query to process top 10 stocks by average price
def process_batch(batch_df, epoch_id):
    # Order by avg_price and limit to top 10
    pandas_df = batch_df.orderBy(F.desc("avg_price")).limit(10).toPandas()

    # Step 3: Create a plot from the Pandas DataFrame
    if not pandas_df.empty:
        # Extract data for plotting
        names = pandas_df["name"]
        avg_prices = pandas_df["avg_price"]

        # Create visualization
        plt.figure(figsize=(12, 6))
        bars = plt.bar(names, avg_prices, color="skyblue")
        plt.title("Top 10 Most Valuable Stocks", fontsize=14, pad=20)
        plt.xlabel("Stock Name", fontsize=12)
        plt.ylabel("Average Price ($)", fontsize=12)
        plt.xticks(rotation=45)

        # Add value labels on top of the bars
        for bar in bars:
            height = bar.get_height()
            plt.text(
                bar.get_x() + bar.get_width() / 2.0,
                height,
                f"${height:.2f}",
                ha="center",
                va="bottom",
            )

        plt.tight_layout()
        plt.show()
    else:
        print("No data available for visualization.")

# Step 4: Write the streaming query
topStocksQuery = windowedDF.writeStream \
    .outputMode("complete") \
    .format("console") \
    .foreachBatch(process_batch) \
    .start()

# Step 5: Await termination (adjust timeout as needed)
topStocksQuery.awaitTermination(timeout=10)
topStocksQuery.stop()


### Task 2: Identify stocks with falling prices


In [ ]:
df.printSchema()


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, col, last
from pyspark.sql.types import StringType, DoubleType, StructType, StructField, TimestampType


# Use a dictionary to hold the state of each stock
stock_states = {}
# Set to hold names of stocks that lost value
stocks_that_lost_value = set()

# Define a class to hold the state of each stock
class StockState:
    def __init__(self, name):
        self.name = name
        self.last_price = None
        self.last_window_end = None
        self.lost_value = False  # Add flag to track if the stock lost value

    def update(self, price, window_end):
        if self.last_window_end is None or window_end > self.last_window_end:
            if self.last_price is not None and price < self.last_price:
                self.lost_value = True # Stock lost value
                stocks_that_lost_value.add(self.name)
            else:
                self.lost_value = False  # Stock did not lose value
            self.last_price = price
            self.last_window_end = window_end
        # Return the state including whether the stock lost value
        return (self.name, self.last_price, self.last_window_end, self.lost_value)


# Define the watermark and window operation on the streaming DataFrame
windowedDF = df \
    .withWatermark("timestamp", "5 minutes") \
    .groupBy(
        col("name"),
        window(col("timestamp"), "10 minutes")
    ) \
    .agg(
        last("price").alias("last_price")
    )

def process_batch(df, epoch_id):
    # Process the DataFrame row by row
    for row in df.collect():
        stock_name = row['name']
        price = row['last_price']
        window_end = row['window'].end

        # Get the stock state or create it if it doesn't exist
        stock_state = stock_states.get(stock_name, StockState(stock_name))
        
        # Update the state and check if the stock lost value
        result = stock_state.update(price, window_end)
        stock_states[stock_name] = stock_state  # Update the state in the dictionary

    print(stocks_that_lost_value)

# Define the streaming query using the process_batch function
query = windowedDF.writeStream \
    .outputMode("update") \
    .foreachBatch(process_batch) \
    .start()  # Start the streaming query

# Await termination of the streaming query (this is running continuously)
query.awaitTermination(timeout=60)
query.stop()


In [ ]:

# Define window parameters
window_duration = "1 minute"
watermark_delay = "2 minutes"

# Create streaming query for stocks losing value
print("Creating streaming query...")
losing_stocks_stream = (df
    .withWatermark("timestamp", watermark_delay)
    .groupBy(
        window(col("timestamp"), window_duration),
        col("name")
    )
    .agg(
        first("price").alias("start_price"),
        last("price").alias("end_price")
    )
    .withColumn("price_change", col("end_price") - col("start_price"))
    .filter(col("price_change") < 0)  # Only keep stocks that lost value
    .orderBy(col("price_change"))  # Order by biggest losses first
)

# Create a list to store the results
collected_data = []

def process_batch(df, epoch_id):
    # Collect data from the current batch
    batch_data = df.collect()
    collected_data.extend(batch_data)

print("Starting streaming query...")
query = (losing_stocks_stream
    .writeStream
    .outputMode("complete")
    .format("console")
    .foreachBatch(process_batch)
    .option("truncate", "false")
    .start()
)

print("Collecting data for 60 seconds...")
time.sleep(60)  # Wait for data collection

print("Analysis Results:")
try:
    if collected_data:
        # Get top 10 biggest losses
        top_10_losses = sorted(collected_data, key=lambda x: x['price_change'])[:10]
        
        # Display table of top 10 losing stocks
        print("\nTop 10 Stocks with Biggest Value Loss:")
        print("=" * 70)
        print(f"{'Stock':<6} {'Start Price($)':<12} {'End Price($)':<12} {'Change($)':<10} {'Change(%)':<10}")
        print("-" * 70)
        for stock in top_10_losses:
            percent_change = (stock['price_change'] / stock['start_price']) * 100
            print(f"{stock['name']:<6} ${stock['start_price']:<11.2f} ${stock['end_price']:<11.2f} ${stock['price_change']:<9.2f} {percent_change:<9.2f}%")
        print("=" * 70)
        
        # Create visualization
        names = [stock['name'] for stock in top_10_losses]
        changes = [stock['price_change'] for stock in top_10_losses]
        
        plt.figure(figsize=(12, 6))
        bars = plt.bar(names, changes, color='red')
        plt.title('Top 10 Stocks with Biggest Value Loss', fontsize=14, pad=20)
        plt.xlabel('Stock Symbol', fontsize=12)
        plt.ylabel('Price Change ($)', fontsize=12)
        plt.xticks(rotation=45)
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width() / 2., height,
                     f'${height:.2f}',
                     ha='center', va='bottom' if height >= 0 else 'top')
        
        plt.tight_layout()
        plt.show()
    else:
        print("No data collected for visualization")

except Exception as e:
    print(f"Error creating visualization: {e}")

finally:
    print("\nStopping query...")
    query.awaitTermination(30)  # Allow some extra time for any final processing
    query.stop()
    print("Query execution completed.")


### Task 3: Identify positive price movements


### Task 3: Compare price changes across windows


### Task 4: Apply a percentage-change threshold


In [ ]:
df.printSchema()

In [ ]:
from pyspark.sql.functions import window, col, first, last
import matplotlib.pyplot as plt

# Defining the windowed data with appropriate watermark
windowed_data = df.withWatermark("timestamp", "5 minutes") \
                  .groupBy(window("timestamp", "5 minutes"), col("name")) \
                  .agg((((first(df.price) - last(df.price)) / first(df.price)) * 100).alias("value_change"))

# Defining the threshold for acceptable percentage change
threshold = 5  # A 5% change threshold is used in this example

# Filtering the data for stocks that did not lose too much value
control_pass = windowed_data.filter(col("value_change") >= -threshold)

# Visualization function
def visualize_batch(batch_df, epoch_id):
    try:
        # Collect data for visualization
        data = batch_df.collect()
        
        if data:
            names = [row["name"] for row in data]
            changes = [row["value_change"] for row in data]

            # Plot the data
            plt.figure(figsize=(10, 6))
            bars = plt.bar(names, changes, color=["green" if change >= 0 else "red" for change in changes])
            plt.xlabel("Stock Name")
            plt.ylabel("Percentage Change")
            plt.title("Stocks with Percentage Changes")
            plt.xticks(rotation=45)

            # Annotate bars with values
            for bar, change in zip(bars, changes):
                plt.text(bar.get_x() + bar.get_width() / 2.0, bar.get_height(), f"{change:.2f}%", ha="center", va="bottom")

            plt.tight_layout()
            plt.show()
        else:
            print("No data to visualize for this batch.")
    except Exception as e:
        print(f"Error during visualization: {e}")

# Starting the streaming query
query = control_pass.writeStream \
                .outputMode("update") \
                .foreachBatch(visualize_batch) \
                .format("console") \
                .start()

query.awaitTermination()


## Optional monitoring: InfluxDB and Grafana



In [ ]:
import hashlib

def generate_window_id(window_start):
    # This function assumes window_start is a string like '2023-10-29 14:40:00'
    # Replace non-numeric characters with nothing to create a numeric string
    numeric_string = window_start.replace('-', '').replace(' ', '').replace(':', '')
    
    # Alternatively, use a hash function to generate a shorter, unique identifier
    # window_id = hashlib.md5(window_start.encode()).hexdigest()[:8]  # Use the first 8 characters

    return numeric_string

In [ ]:
import os
import influxdb_client
from influxdb_client.client.write_api import SYNCHRONOUS
from pyspark.sql import Row
import time

class InfluxDBForeachWriter:
    def open(self, partition_id, epoch_id):
        self.url = os.getenv("INFLUXDB_URL", "http://localhost:8086")
        self.token = os.getenv("INFLUXDB_TOKEN", "")
        self.org = os.getenv("INFLUXDB_ORG", "YOUR_INFLUXDB_ORG")
        self.bucket = os.getenv("INFLUXDB_BUCKET", "stocks")
        self.client = influxdb_client.InfluxDBClient(
            url=self.url,
            token=self.token,
            org=self.org
        )
        self.write_api = self.client.write_api(write_options=SYNCHRONOUS)
        return True

    def process(self, row: Row,window_id:int):
        try:
            # Convert timestamp to the correct format for InfluxDB (RFC3339)
            # Assuming timestamp is a string in the format 'YYYY-MM-DD HH:MM:SS'

            window_start_str = row.window.start.strftime('%Y-%m-%d %H:%M:%S')

            # Prepare data for InfluxDB
            data_point = influxdb_client.Point("stock_testing") \
                .tag("name", str(row.name)) \
                .tag("window_id",window_id) \
                .field("avg_price", float(row.avg_price)) \
                .field("window_start", window_start_str) \
                .time(int(time.time_ns()))  # Use the current time in nanoseconds
            print(f"Attempting to write data point: {data_point.to_line_protocol()}")

            # Write data to InfluxDB
            self.write_api.write(bucket=self.bucket, record=data_point)
            print("Write successful")
        except Exception as e:
            print(f"An error occurred: {e}")



    def close(self, error):
        # Close the InfluxDB client
        self.write_api.close()
        self.client.close()
        return True


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, col
from pyspark.sql import functions as F
from pyspark.sql.streaming import DataStreamWriter



# Initialize the window_id globally
window_id = 0

def write_to_influxdb(df, epoch_id):
    # Use the global keyword to indicate that you are using the global window_id variable
    global window_id

    # Instantiate the InfluxDBForeachWriter outside the loop
    writer = InfluxDBForeachWriter()

    if writer.open(epoch_id, None):  # Open the connection to InfluxDB using the epoch_id
        print(window_id)
        # Sort by average price and take the top 5
        top5_df = df.orderBy(F.desc("avg_price")).limit(5).collect()

        for row in top5_df:
            try:
                # Pass window_id along with the row data
                writer.process(row, window_id)
            except Exception as e:
                print(f"Error writing to InfluxDB: {e}")
                break  # or continue, depending on your requirement

        # Increment window_id for the next batch of data
        window_id += 1
        writer.close(epoch_id)  # Close the connection after all rows are written
    else:
        print("Failed to open InfluxDB connection")




writer = InfluxDBForeachWriter()

windowedDF = df.groupBy(
    window(col("timestamp"), "20 minutes"),
    "name"
).avg("price").withColumnRenamed("avg(price)", "avg_price")

# Define the streaming query using foreachBatch
topStocksQuery = windowedDF \
    .writeStream \
    .outputMode("complete") \
    .foreachBatch(write_to_influxdb) \
    .start()

topStocksQuery.awaitTermination(timeout=50)
topStocksQuery.stop()
